# UNet 分片加载：默认不下载、不加载

这是默认关闭传输的入口。不要在浏览器里改环境变量。
本次已授权的 GPU 会话使用 `kaggle_unet_load_enabled.ipynb`，那里的开关已经写成 True。
只下载官方 UNet 一个文件。`/kaggle/working` 比该文件小，脚本会改选 `/tmp`；空间不够就停止。
加载只调用真实 worker 的 `load_unet`，然后关闭。不编码文本，不解码 VAE，不采样。
总预算 1080 秒。下载如果按已用速度无法在留下的加载时间内完成，就取消这一次，不重试。
可用内存低于 2GiB 或阶段超时会写 evidence zip、sha256 和 manifest。脚本自己退出。浏览器里仍需要点 Stop。


In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

ENTRY_SCRIPT = Path("scripts") / "run_actor_session.py"
MARKER_SCRIPT = Path("scripts") / "bootstrap_actor_admission.py"
MAX_DEPTH = 6


def default_seeds() -> list[Path]:
    return [Path.cwd(), Path("/kaggle/input"), Path("/kaggle/working")]


def find_package_root(seeds: list[Path] | None = None, environ: dict[str, str] | None = None) -> Path:
    """Return the only directory that contains the current admission entry.

    ``H3_T4_PACKAGE_ROOT`` selects one directory and disables the search.
    A modern Kaggle mount is ``/kaggle/input/datasets/<owner>/<dataset>/<package>``.
    Older sessions mounted the package directly under ``/kaggle/input`` or
    ``/kaggle/working``. More than one match is an error.
    """

    env = os.environ if environ is None else environ
    pinned = env.get("H3_T4_PACKAGE_ROOT", "").strip()
    if pinned:
        root = Path(pinned)
        if not _is_admission_package(root):
            raise SystemExit(f"H3_T4_PACKAGE_ROOT is not the admission package: {root}")
        return root.resolve()
    found: list[Path] = []
    seen: set[str] = set()
    for seed in seeds if seeds is not None else default_seeds():
        if not seed.exists():
            continue
        for match in _scan(seed.resolve(), MAX_DEPTH):
            key = str(match.resolve())
            if key not in seen:
                seen.add(key)
                found.append(match.resolve())
    if not found:
        raise SystemExit(
            "admission package not found. Upload the folder that contains "
            "scripts/run_actor_session.py or set H3_T4_PACKAGE_ROOT."
        )
    if len(found) > 1:
        lines = "\n".join(str(path) for path in found)
        raise SystemExit("multiple admission packages found; set H3_T4_PACKAGE_ROOT to one:\n" + lines)
    return found[0]


def _is_admission_package(root: Path) -> bool:
    return (root / ENTRY_SCRIPT).is_file() and (root / MARKER_SCRIPT).is_file()


def _scan(seed: Path, max_depth: int) -> list[Path]:
    matches: list[Path] = []
    for current, dirnames, _filenames in os.walk(seed):
        current_path = Path(current)
        try:
            depth = len(current_path.relative_to(seed).parts)
        except ValueError:
            depth = max_depth + 1
        if depth > max_depth:
            dirnames.clear()
            continue
        if _is_admission_package(current_path):
            matches.append(current_path)
            dirnames.clear()
    return matches

root = find_package_root()
evidence = Path("/kaggle/working/h3-unet-load")
evidence.mkdir(parents=True, exist_ok=True)
enable = os.environ.get("H3_T4_ENABLE_TRANSFER", "0") == "1"
command = [sys.executable, str(root / "scripts" / "run_unet_load_session.py"), "--evidence-dir", str(evidence), "--budget-s", "1080", "--comfy-dir", "/tmp/ComfyUI-h3"]
if enable:
    command.append("--enable-transfer")
print("enable_transfer", enable)
code = subprocess.call(command, cwd=str(root))
print("process_exit", code)
spec = importlib.util.spec_from_file_location("h3_unet_session", root / "scripts" / "run_unet_load_session.py")
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
module.show_session_evidence(evidence)
